# Base Installer
> Abstract base class for MCP installers

In [ ]:
#| default_exp install.base

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

Types and I/O utilities come from `utils`.

In [ ]:
#| export
from __future__ import annotations
import shutil
import subprocess
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Optional, Protocol, runtime_checkable

# Re-export from utils for convenience
from nbdev_mcp.utils.types import (
    InstallResult, InstallAction, MCPServerConfig, get_python_path
)
from nbdev_mcp.utils.io import (
    read_json_config, write_json_config,
    read_toml_config, write_toml_config,
    open_file_in_editor
)
from nbdev_mcp.utils.paths import (
    get_vscode_mcp_path, get_vscode_user_dir,
    get_claude_config_path, get_claude_config_dir,
    get_codex_config_path, get_codex_config_dir,
    get_ollama_config_path, get_ollama_config_dir,
    get_all_config_paths, get_mcp_config_path
)

## Command Utilities

In [ ]:
#| export
def which(cmd: str) -> Optional[str]:
    """Find an executable in PATH."""
    return shutil.which(cmd)

In [ ]:
#| export
def run_command(
    args: list[str],
    check: bool = False,
    capture: bool = True
) -> subprocess.CompletedProcess:
    """Run a command and return the result.
    
    Parameters
    ----------
    args : list[str]
        Command and arguments.
    check : bool
        Raise on non-zero exit.
    capture : bool
        Capture stdout/stderr.
    
    Returns
    -------
    subprocess.CompletedProcess
    """
    return subprocess.run(args, capture_output=capture, text=True, check=check)

## Installer Protocol

In [ ]:
#| export
@runtime_checkable
class Installer(Protocol):
    """Protocol for MCP installers."""
    
    @property
    def name(self) -> str: ...
    
    @property
    def config_path(self) -> Path: ...
    
    def is_configured(self) -> bool: ...
    
    def install(self, server: MCPServerConfig, force: bool = False, dry_run: bool = False) -> InstallResult: ...
    
    def uninstall(self, dry_run: bool = False) -> InstallResult: ...

## Base Installer

In [ ]:
#| export
class BaseInstaller(ABC):
    """Abstract base class for installers.
    
    Subclasses must implement:
    - name: Provider name property
    - config_path: Config file path property  
    - is_configured(): Check if nbdev is configured
    - do_install(): Perform installation
    - do_uninstall(): Perform uninstallation
    
    Note: No private attributes (_underscore). All state is explicit.
    """
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Provider name."""
        pass
    
    @property
    @abstractmethod
    def config_path(self) -> Path:
        """Path to configuration file."""
        pass
    
    @abstractmethod
    def is_configured(self) -> bool:
        """Check if nbdev is already configured."""
        pass
    
    @abstractmethod
    def do_install(self, server: MCPServerConfig) -> None:
        """Perform the actual installation."""
        pass
    
    @abstractmethod
    def do_uninstall(self) -> None:
        """Perform the actual uninstallation."""
        pass
    
    def install(self, server: MCPServerConfig, force: bool = False, dry_run: bool = False) -> InstallResult:
        """Install the MCP server configuration."""
        if self.is_configured() and not force:
            return InstallResult(
                success=True,
                message=f"{self.name}: Already configured. Use --force to overwrite.",
                config_path=self.config_path,
                action='skipped',
                provider=self.name
            )
        
        if dry_run:
            return InstallResult(
                success=True,
                message=f"{self.name}: Would configure at {self.config_path}",
                config_path=self.config_path,
                action='dry_run',
                provider=self.name
            )
        
        try:
            existed = self.config_path.exists()
            self.do_install(server)
            action = 'updated' if existed else 'created'
            return InstallResult(
                success=True,
                message=f"{self.name}: Configuration {action}",
                config_path=self.config_path,
                action=action,
                provider=self.name
            )
        except Exception as e:
            return InstallResult(
                success=False,
                message=f"{self.name}: Installation failed - {e}",
                config_path=self.config_path,
                action='error',
                provider=self.name,
                details={'error': str(e)}
            )
    
    def uninstall(self, dry_run: bool = False) -> InstallResult:
        """Remove the MCP server configuration."""
        if not self.is_configured():
            return InstallResult(
                success=True,
                message=f"{self.name}: Not configured, nothing to remove.",
                config_path=self.config_path,
                action='skipped',
                provider=self.name
            )
        
        if dry_run:
            return InstallResult(
                success=True,
                message=f"{self.name}: Would remove from {self.config_path}",
                config_path=self.config_path,
                action='dry_run',
                provider=self.name
            )
        
        try:
            self.do_uninstall()
            return InstallResult(
                success=True,
                message=f"{self.name}: Configuration removed",
                config_path=self.config_path,
                action='updated',
                provider=self.name
            )
        except Exception as e:
            return InstallResult(
                success=False,
                message=f"{self.name}: Uninstall failed - {e}",
                config_path=self.config_path,
                action='error',
                provider=self.name,
                details={'error': str(e)}
            )
    
    def status(self) -> dict:
        """Get installation status."""
        return {
            'provider': self.name,
            'exists': self.config_path.exists(),
            'configured': self.is_configured(),
            'path': str(self.config_path)
        }

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()